In [10]:
!pip install transformers

In [9]:
import gradio as gr
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Загружаем модель
print("Загружаем модель Trash-Net...")
classifier = pipeline("image-classification", model="prithivMLmods/Trash-Net")
print("Модель готова к работе!")

# Функция для обработки изображения
def analyze_waste(image_input, url_input):
    try:
        # Определяем, откуда брать изображение
        if image_input is not None:
            # Если загружено через файл
            if isinstance(image_input, str):
                # Если это путь к файлу
                image = Image.open(image_input)
            else:
                # Если это уже изображение PIL
                image = image_input
            source = "загруженного файла"
        elif url_input and url_input.strip():
            # Если введена ссылка
            response = requests.get(url_input.strip(), stream=True)
            image = Image.open(response.raw)
            source = "URL-ссылки"
        else:
            return "Пожалуйста, загрузите фото или введите ссылку!", None

        # Анализируем изображение
        results = classifier(image)

        # Формируем текстовый результат
        text_result = f"**Результаты анализа {source}:**\n\n"
        for i, result in enumerate(results, 1):
            text_result += f"{i}. **{result['label']}**: {result['score']*100:.2f}%\n"

        # Добавляем рекомендацию по сортировке
        main_category = results[0]['label'].lower()
        confidence = results[0]['score']*100

        recycling_tips = {
            'plastic': "♻️ **Пластик**: Можно переработать. Снимите крышку, сполосните и сдайте в контейнер для пластика",
            'paper': "📄 **Бумага**: Можно переработать. Сдайте в макулатуру (чистую, без скрепок и пластика)",
            'cardboard': "📦 **Картон**: Можно переработать. Сдайте в макулатуру (лучше сложить компактно)",
            'glass': "🥛 **Стекло**: Можно переработать. Сдайте в стеклотару (без крышек и этикеток)",
            'metal': "🥫 **Металл**: Можно переработать. Сдайте в металлолом (алюминий, жесть)",
            'trash': "🗑️ **Смешанные отходы**: Отправляйте в общий контейнер. Возможно, это неперерабатываемый мусор"
        }

        tip = recycling_tips.get(main_category,
                                 "❓ Тип отходов не определен, проверьте местные правила сортировки")

        text_result += f"\n---\n### 🌱 Рекомендация:\n{tip}\n"
        text_result += f"*\nУверенность модели: {confidence:.1f}%*"

        return text_result, image

    except Exception as e:
        return f"❌ Ошибка: {str(e)}\n\nПопробуйте другую ссылку или загрузите фото напрямую.", None

# Создаем интерфейс
with gr.Blocks(title="Эко-помощник по сортировке мусора", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🌍 Эко-помощник по сортировке мусора

    Загрузите фото отходов, и нейросеть определит их тип и подскажет, как правильно сортировать!
    Модель обучена на датасете TrashNet и умеет распознавать: **пластик, бумагу, картон, стекло, металл и смешанные отходы**.

    ---
    """)

    with gr.Row():
        with gr.Column(scale=1):
            # Загрузка файла
            file_input = gr.Image(type="pil", label="📁 Загрузить фото с компьютера",
                                 height=300)

        with gr.Column(scale=1):
            # Ввод URL
            url_input = gr.Textbox(label="🔗 Или вставьте ссылку на фото",
                                   placeholder="https://example.com/photo.jpg",
                                   lines=2)

    # Кнопка анализа
    analyze_btn = gr.Button("🔍 Анализировать мусор", variant="primary", size="lg")

    with gr.Row():
        # Результат
        with gr.Column(scale=1):
            output_image = gr.Image(label="🖼️ Анализируемое изображение", height=300)
        with gr.Column(scale=1):
            output_text = gr.Markdown(label="📊 Результаты анализа")

    # Обработчик нажатия кнопки
    analyze_btn.click(
        fn=analyze_waste,
        inputs=[file_input, url_input],
        outputs=[output_text, output_image]
    )

    # Примеры для тестирования (исправленная версия)
    gr.Markdown("### 📸 Попробуйте эти примеры:")
    gr.Markdown("""
    **Скопируйте одну из ссылок и вставьте в поле выше:**

    1. 🥤 **Пластик**: `https://ir.ozone.ru/s3/multimedia-1-p/7641592297.jpg`
    2. 📄 **Бумага**: `https://img.ixbt.site/live/topics/preview/00/07/64/21/708348f064.jpg`
    3. 📦 **Картон**: `https://avatars.mds.yandex.net/i?id=650416e1ffc08072fbc8a59d9a6c0221_l-9227066-images-thumbs&n=13`
    4. 🥫 **Металл**: `https://as2.ftcdn.net/jpg/00/60/98/17/1000_F_60981742_6yBb4TkWkzWUiLFEsLEtLptrpGtx7y53.jpg`
    """)

    # Добавляем кнопки для быстрой вставки примеров
    with gr.Row():
        plastic_btn = gr.Button("🥤 Пластик")
        paper_btn = gr.Button("📄 Бумага")
        metal_btn = gr.Button("🥫 Металл")

    # Функция для вставки примеров
    def set_example(url):
        return url

    plastic_btn.click(fn=lambda: "https://ir.ozone.ru/s3/multimedia-1-p/7641592297.jpg", outputs=[url_input])
    paper_btn.click(fn=lambda: "https://img.ixbt.site/live/topics/preview/00/07/64/21/708348f064.jpg", outputs=[url_input])
    metal_btn.click(fn=lambda: "https://as2.ftcdn.net/jpg/00/60/98/17/1000_F_60981742_6yBb4TkWkzWUiLFEsLEtLptrpGtx7y53.jpg", outputs=[url_input])

    gr.Markdown("""
    ---
    ### 📝 Как пользоваться:
    1. **Загрузите фото** через кнопку "Загрузить фото с компьютера" или **вставьте ссылку** на изображение
    2. Нажмите кнопку "Анализировать мусор"
    3. Получите результат с рекомендацией по сортировке

    ### ⚠️ Примечание:
    - Для лучших результатов фото должно быть четким, мусор должен занимать бóльшую часть кадра
    - Модель работает лучше с фото одного предмета, чем с кучей мусора
    """)

# Запускаем интерфейс
print("🚀 Запускаем интерфейс...")
print("После запуска нажмите на появившуюся ссылку ↓")
demo.launch(share=True, debug=False)

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 80, in main
    return command.main(cmd_args)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 100, in main
    return self._main(args)
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 118, in _main
    level_number = setup_logging(
                   ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/utils/logging.py", line 274, in setup_logging
    logging.config.dictConfig(
  File "/usr/lib/python3.12/logging/config.py", line 942, in dictConfig
    dictConfigClass(config).configure()
  File "/usr/lib/python3.12/logging/config.py", line 680, in configure
    _handle_existing_loggers(existing, child_loggers,
  File 

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Модель готова к работе!


/tmp/ipython-input-2481556050.py:70: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Эко-помощник по сортировке мусора", theme=gr.themes.Soft()) as demo:


🚀 Запускаем интерфейс...
После запуска нажмите на появившуюся ссылку ↓
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ef67d8ee43f1c2069.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
